# WORKING WITH DIFFERENT TYPES OF DATA
    We also review working with a variety of different kinds of data, including the following:

    - Booleans

    - Numbers

    - Strings

    - Dates and timestamps

    - Handling null

    - Complex types

    - User-defined functions


In [4]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('Data').getOrCreate()
df=spark.read.format('csv').load('file:///home/unik/PYSPARK/by-day/2010-12-01.csv',header=True,inferSchema=True);

In [5]:
df.printSchema()
df.createOrReplaceTempView('dfTable')

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



# WORKING WITH BOOLEANS

In [6]:
from pyspark.sql.functions import col
df.where(col("InvoiceNo") != 536365)\
  .select("InvoiceNo", "Description")\
  .show(5, False)

+---------+-----------------------------+
|InvoiceNo|Description                  |
+---------+-----------------------------+
|536366   |HAND WARMER UNION JACK       |
|536366   |HAND WARMER RED POLKA DOT    |
|536367   |ASSORTED COLOUR BIRD ORNAMENT|
|536367   |POPPY'S PLAYHOUSE BEDROOM    |
|536367   |POPPY'S PLAYHOUSE KITCHEN    |
+---------+-----------------------------+
only showing top 5 rows



In [7]:
df.where("InvoiceNo = 536365").show(5, False)

+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |2010-12-01 08:26:00|2.55     |17850.0   |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |2010-12-01 08:26:00|2.75     |17850.0   |United Kingdom|
|536365   |84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
|536365   |84029E   |RED WOOLLY HOTTIE WHITE HEART.     |6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
+---------+-----

# WORKING WITH NUMBERS

In [8]:
from pyspark.sql.functions import expr,pow
fabricatedQuantity=pow(col('Quantity')*col('UnitPrice'),2)+5
df.select(expr('CustomerID'),fabricatedQuantity.alias('realQuantity')).show(5)

+----------+------------------+
|CustomerID|      realQuantity|
+----------+------------------+
|   17850.0|239.08999999999997|
|   17850.0|          418.7156|
|   17850.0|             489.0|
|   17850.0|          418.7156|
|   17850.0|          418.7156|
+----------+------------------+
only showing top 5 rows



In [9]:
df.selectExpr(
  "CustomerId",
  "(POWER((Quantity * UnitPrice), 2.0) + 5) as realQuantity").show(2)

+----------+------------------+
|CustomerId|      realQuantity|
+----------+------------------+
|   17850.0|239.08999999999997|
|   17850.0|          418.7156|
+----------+------------------+
only showing top 2 rows



In [15]:
# ROUNDING AND BROUNDING
from pyspark.sql.functions import round,bround
df.select(round(lit('2.5')),bround(lit(2.5))).show(2)

+-------------+--------------+
|round(2.5, 0)|bround(2.5, 0)|
+-------------+--------------+
|          3.0|           2.0|
|          3.0|           2.0|
+-------------+--------------+
only showing top 2 rows



In [17]:
# TO FIND CORRELATION AMONG TWO VALUES
from pyspark.sql.functions import corr
df.stat.corr('QUANTITY','UNITPRICE')
df.select(corr('QUANTITY','UNITPRICE').alias('CORRELATION')).show()

+--------------------+
|         CORRELATION|
+--------------------+
|-0.04112314436835551|
+--------------------+



In [21]:
df.describe().show();
df.select('Country').distinct().count();

+-------+-----------------+------------------+--------------------+------------------+------------------+------------------+--------------+
|summary|        InvoiceNo|         StockCode|         Description|          Quantity|         UnitPrice|        CustomerID|       Country|
+-------+-----------------+------------------+--------------------+------------------+------------------+------------------+--------------+
|  count|             3108|              3108|                3098|              3108|              3108|              1968|          3108|
|   mean| 536516.684944841|27834.304044117645|                NULL| 8.627413127413128| 4.151946589446603|15661.388719512195|          NULL|
| stddev|72.89447869788873|17407.897548583845|                NULL|26.371821677029203|15.638659854603892|1854.4496996893627|          NULL|
|    min|           536365|             10002| 4 PURPLE FLOCK D...|               -24|               0.0|           12431.0|     Australia|
|    max|          C

In [20]:
df.groupBy('Country','Description').agg(
    sum('Quantity').alias('Total Quantity'),\
    max('Quantity').alias('MAX QUANTITY')
).show()

TypeError: unsupported operand type(s) for +: 'int' and 'str'

# WORKING WITH STRINGS

In [22]:
# To change the first word to UpperCase
from pyspark.sql.functions import initcap
df.select(initcap(col('Description'))).show(5)

+--------------------+
|initcap(Description)|
+--------------------+
|White Hanging Hea...|
| White Metal Lantern|
|Cream Cupid Heart...|
|Knitted Union Fla...|
|Red Woolly Hottie...|
+--------------------+
only showing top 5 rows



In [23]:
# To make all words in upper and lower 
from pyspark.sql.functions import lower, upper
df.select(col("Description"),
lower(col("Description")),
upper(lower(col("Description")))).show(5)

+--------------------+--------------------+-------------------------+
|         Description|  lower(Description)|upper(lower(Description))|
+--------------------+--------------------+-------------------------+
|WHITE HANGING HEA...|white hanging hea...|     WHITE HANGING HEA...|
| WHITE METAL LANTERN| white metal lantern|      WHITE METAL LANTERN|
|CREAM CUPID HEART...|cream cupid heart...|     CREAM CUPID HEART...|
|KNITTED UNION FLA...|knitted union fla...|     KNITTED UNION FLA...|
|RED WOOLLY HOTTIE...|red woolly hottie...|     RED WOOLLY HOTTIE...|
+--------------------+--------------------+-------------------------+
only showing top 5 rows



# REGULAR EXPRESSION
    -Probably one of the most frequently performed tasks is searching for the existence of one string in another or replacing all mentions of a string with another value this is often done by regular expressions
    

In [24]:
from pyspark.sql.functions import regexp_replace
regex_string = "BLACK|WHITE|RED|GREEN|BLUE"
df.select(regexp_replace(col("Description"),regex_string,"COLOR").alias("color_clean"),col("Description")).show(2)

+--------------------+--------------------+
|         color_clean|         Description|
+--------------------+--------------------+
|COLOR HANGING HEA...|WHITE HANGING HEA...|
| COLOR METAL LANTERN| WHITE METAL LANTERN|
+--------------------+--------------------+
only showing top 2 rows



In [25]:
from pyspark.sql.functions import translate
df.select(translate(col("Description"),"LEET","1337"),col("Description")).show(2,truncate=False)

+----------------------------------+----------------------------------+
|translate(Description, LEET, 1337)|Description                       |
+----------------------------------+----------------------------------+
|WHI73 HANGING H3AR7 7-1IGH7 HO1D3R|WHITE HANGING HEART T-LIGHT HOLDER|
|WHI73 M37A1 1AN73RN               |WHITE METAL LANTERN               |
+----------------------------------+----------------------------------+
only showing top 2 rows



In [26]:
from pyspark.sql.functions import regexp_extract
extract_str = "(BLACK|WHITE|RED|GREEN|BLUE)"
df.select(regexp_extract(col("Description"),extract_str,1).alias("color_clean"),col("Description")).show(2,truncate=False)

+-----------+----------------------------------+
|color_clean|Description                       |
+-----------+----------------------------------+
|WHITE      |WHITE HANGING HEART T-LIGHT HOLDER|
|WHITE      |WHITE METAL LANTERN               |
+-----------+----------------------------------+
only showing top 2 rows



    Note:Sometimes, rather than extracting values, we simply want to check for their existence. We can do this with the contains method on each column. This will return a Boolean declaring whether the value you specify is in the column’s string:

In [28]:
from pyspark.sql.functions import instr
containsBlack = instr(col("Description"), "BLACK") >= 1
df.withColumn("hasSimpleColor", containsBlack)\
  .where("hasSimpleColor")\
  .select("Description").show(3, False)

+---------------------------------+
|Description                      |
+---------------------------------+
|JUMBO  BAG BAROQUE BLACK WHITE   |
|EDWARDIAN PARASOL BLACK          |
|WOOD BLACK BOARD ANT WHITE FINISH|
+---------------------------------+
only showing top 3 rows



# WORKING WITH DATE AND TIMESTAMPS
    -Working with dates and timestamps closely relates to working with strings because we often store our timestamps or dates as strings and convert them into date types at runtime. 

In [29]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [30]:
from pyspark.sql.functions import current_date, current_timestamp
dateDF=spark.range(10)\
  .withColumn('Today',current_date())\
  .withColumn('Now',current_timestamp())
dateDF.createOrReplaceTempView('dateTable')

In [31]:
dateDF.printSchema()

root
 |-- id: long (nullable = false)
 |-- Today: date (nullable = false)
 |-- Now: timestamp (nullable = false)



In [32]:
from pyspark.sql.functions import date_add, date_sub
dateDF.select(date_sub(col("today"),5),date_add(col("today"),5)).show(1)

+------------------+------------------+
|date_sub(today, 5)|date_add(today, 5)|
+------------------+------------------+
|        2026-03-27|        2026-04-06|
+------------------+------------------+
only showing top 1 row



# WORKING WITH NULLS IN DATA
    -The primary way of interacting with null values, at DataFrame scale, is to use the .na subpackage on a DataFrame. 
    -There are two things you can do with null values: you can explicitly drop nulls or you can fill them with a value (globally or on a per-column basis). 

## COALESCE

In [33]:
from pyspark.sql.functions import coalesce
df.select(coalesce(col('Description'),col('CustomerID'))).show(5,truncate=False)

+-----------------------------------+
|coalesce(Description, CustomerID)  |
+-----------------------------------+
|WHITE HANGING HEART T-LIGHT HOLDER |
|WHITE METAL LANTERN                |
|CREAM CUPID HEARTS COAT HANGER     |
|KNITTED UNION FLAG HOT WATER BOTTLE|
|RED WOOLLY HOTTIE WHITE HEART.     |
+-----------------------------------+
only showing top 5 rows



## DROP
    -Specifying "any" as an argument drops a row if any of the values are null. Using “all” drops the row only if all values are null or NaN for that row


In [34]:
df.na.drop()
df.na.drop("any")
df.na.drop('all')

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: timestamp, UnitPrice: double, CustomerID: double, Country: string]

    - We can also apply this to certain sets of columns by passing in an array of columns:

In [35]:
df.na.drop("all",subset=["StockCode","InvoiceNo"])

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: timestamp, UnitPrice: double, CustomerID: double, Country: string]

## FILL
    -Using the fill function, you can fill one or more columns with a set of values. This can be done by specifying a map—​that is a particular value and a set of columns.

In [36]:
df.na.fill("Unknown")

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: timestamp, UnitPrice: double, CustomerID: double, Country: string]

    -We could do the same for columns of type Integer by using df.na.fill(5:Integer), or for Doubles df.na.fill(5:Double)

In [37]:
fill_cols_vals = {"StockCode":5,"Description":"No Value"}
df.na.fill(fill_cols_vals)

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: timestamp, UnitPrice: double, CustomerID: double, Country: string]

# REPLACE
    -Probably the most common use case is to replace all values in a certain column according to their current value.
    -The only requirement is that this value be the same type as the original value.

In [38]:
df.na.replace([""],["UNKNOWN"],'Description')

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: timestamp, UnitPrice: double, CustomerID: double, Country: string]

# ORDERING
    -As we discussed we can use asc_nulls_first, desc_nulls_first, asc_nulls_last, or desc_nulls_last to specify where you would like your null values to appear in an ordered DataFrame.

# WORKING WITH COMPLEX TYPES
    -Complex types can help you organize and structure your data in ways that make more sense for the problem that you are hoping to solve. 
    -There are three kinds of complex types: structs, arrays, and maps.

## STRUCTS
    -We can think of structs as DataFrames within DataFrames
    -We can create a struct by wrapping a set of columns in parenthesis in a query

In [41]:
from pyspark.sql.functions import struct
complexDF = df.select(struct("Description","InvoiceNo").alias("complex"))
complexDF.createOrReplaceTempView("complexDF")

    -We can query it just as we might another DataFrame, the only difference is that we use a dot syntax to do so, or the column method getField.

In [43]:
complexDF.select('complex.Description')
complexDF.select(col('complex').getField('Description'))

DataFrame[complex.Description: string]

## ARRAYS

### SPLIT

In [45]:
from pyspark.sql.functions import split
df.select(split(col('Description')," ")).show(2,truncate=False)

+----------------------------------------+
|split(Description,  , -1)               |
+----------------------------------------+
|[WHITE, HANGING, HEART, T-LIGHT, HOLDER]|
|[WHITE, METAL, LANTERN]                 |
+----------------------------------------+
only showing top 2 rows



### ARRAY LENGTH

In [46]:
from pyspark.sql.functions import size
df.select(size(split(col('Description')," "))).show(2,truncate=False)

+-------------------------------+
|size(split(Description,  , -1))|
+-------------------------------+
|5                              |
|3                              |
+-------------------------------+
only showing top 2 rows



### ARRAY CONTAINS

In [49]:
from pyspark.sql.functions import array_contains
df.select(array_contains(split(col('Description')," "),"WHITE")).show(5,truncate=False)

+------------------------------------------------+
|array_contains(split(Description,  , -1), WHITE)|
+------------------------------------------------+
|true                                            |
|true                                            |
|false                                           |
|false                                           |
|true                                            |
+------------------------------------------------+
only showing top 5 rows



### MAPS
    -Maps are created by using the map function and key-value pairs of columns. You then can select them just like you might select from an array.

In [53]:
from pyspark.sql.functions import create_map
df.select(create_map(col('Description'),col('InvoiceNo')).alias('complex_map')).show(2,truncate=False)

+----------------------------------------------+
|complex_map                                   |
+----------------------------------------------+
|{WHITE HANGING HEART T-LIGHT HOLDER -> 536365}|
|{WHITE METAL LANTERN -> 536365}               |
+----------------------------------------------+
only showing top 2 rows



# USER DEFINED FUNCTIONS
    -User-defined functions (UDFs) make it possible for you to write your own custom transformations using Python or Scala and even use external libraries. 

In [55]:
from pyspark.sql.functions import udf
udfExampleDF=spark.range(5).toDF("num")
def power3(double_value):
    return double_value**3
udfExampleDF.select(power3(col('num'))).show(4)

+-------------+
|POWER(num, 3)|
+-------------+
|          0.0|
|          1.0|
|          8.0|
|         27.0|
+-------------+
only showing top 4 rows



In [56]:
df.show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|   17850.0|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|   17850.0|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|2010-12-01 08:26:00|     7.65|   17850.0|United Kingdom|
|   536365|    21730|GLASS S

In [74]:
def unit_price(value):
    return value*0.1+value
unit_udf=udf(unit_price,IntegerType())
df.select(unit_price(col('UnitPrice'))).show(5)

+-------------------------------+
|((UnitPrice * 0.1) + UnitPrice)|
+-------------------------------+
|             2.8049999999999997|
|                          3.729|
|                          3.025|
|                          3.729|
|                          3.729|
+-------------------------------+
only showing top 5 rows



In [73]:
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import col
def quantity_filter(value):
    if value > 5:
        return value
    else:
        return None
quantity_udf = udf(quantity_filter, IntegerType())
df.select(quantity_udf(col('Quantity')).alias("Filtered_Quantity")).show(5)

+-----------------+
|Filtered_Quantity|
+-----------------+
|                6|
|                6|
|                8|
|                6|
|                6|
+-----------------+
only showing top 5 rows



In [86]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import DoubleType
def filter_unit(value):
    if value is not None and value > 2:
        return value
    else:
        return None
filter_udf = udf(filter_unit, DoubleType())
df.select(
    filter_udf(col('UnitPrice')).alias('Filtered_Price')).show()

+--------------+
|Filtered_Price|
+--------------+
|          2.55|
|          3.39|
|          2.75|
|          3.39|
|          3.39|
|          7.65|
|          4.25|
|          NULL|
|          NULL|
|          NULL|
|           2.1|
|           2.1|
|          3.75|
|          NULL|
|          4.25|
|          4.95|
|          9.95|
|          5.95|
|          5.95|
|          7.95|
+--------------+
only showing top 20 rows



# TRANSFORM()

In [91]:
def filter_value(value,precent):
    return df.withColumn("Filtered Value",df.UnitPrice*precent+df.UnitPrice)
df.transform(filter_value,0.1).show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|    Filtered Value|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|   17850.0|United Kingdom|2.8049999999999997|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|             3.729|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|   17850.0|United Kingdom|             3.025|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|             3.729|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|    